In [1]:
# DEPENDENCIES

import sys
sys.path.insert(0, '../..')
from dependencies import *
from constants import *
from paths import *
import helper_functions

In [2]:
# SETUP

FUGLSANG_SUBJECTS = helper_functions.fuglsang_get_subjects()
# SNHL_SUBJECTS_NH = helper_functions.snhl_get_subjects(group='NH')
# SNHL_SUBJECTS_HI = helper_functions.snhl_get_subjects(group='HI')

In [3]:
# HELPER FUNCTIONS

def report_binomial(label, correct, n, chance=0.5):
    result = binomtest(int(correct), n, chance, alternative='greater')
    print(f"{label}")
    print(f"  Accuracy: {correct}/{n} ({correct/n:.1%})")
    print(f"  Binomial test vs chance: p = {result.pvalue:.4f}"
          + (" *" if result.pvalue < 0.05 else ""))
    print()

def report_ttest(label, r_att, r_ign):
    r_att = np.array(list(r_att.values()) if isinstance(r_att, dict) else r_att)
    r_ign = np.array(list(r_ign.values()) if isinstance(r_ign, dict) else r_ign)
    t, p = ttest_rel(r_att, r_ign)
    d = (r_att - r_ign).mean() / (r_att - r_ign).std()
    print(f"{label}")
    print(f"  r_att: {r_att.mean():.4f} ± {r_att.std():.4f}")
    print(f"  r_ign: {r_ign.mean():.4f} ± {r_ign.std():.4f}")
    print(f"  Paired t-test: t = {t:.3f}, p = {p:.4f}"
          + (" *" if p < 0.05 else ""))
    print(f"  Cohen's d = {d:.3f}")
    print()

def report_ttest_between(label, r_a, r_b):
    r_a = np.array(list(r_a.values()) if isinstance(r_a, dict) else r_a)
    r_b = np.array(list(r_b.values()) if isinstance(r_b, dict) else r_b)
    t, p = ttest_ind(r_a, r_b)
    print(f"{label}")
    print(f"  Group A: {r_a.mean():.4f} ± {r_a.std():.4f} (n={len(r_a)})")
    print(f"  Group B: {r_b.mean():.4f} ± {r_b.std():.4f} (n={len(r_b)})")
    print(f"  Independent t-test: t = {t:.3f}, p = {p:.4f}"
          + (" *" if p < 0.05 else ""))
    print()

def report_mcnemar(label, decisions_a, decisions_b):
    da = np.array(list(decisions_a.values()) if isinstance(decisions_a, dict) else decisions_a)
    db = np.array(list(decisions_b.values()) if isinstance(decisions_b, dict) else decisions_b)
    table = np.array([
        [np.sum( da &  db), np.sum( da & ~db)],
        [np.sum(~da &  db), np.sum(~da & ~db)]
    ])
    result = mcnemar(table, exact=True)
    print(f"{label}")
    print(f"  Contingency table:\n    {table[0].tolist()}\n    {table[1].tolist()}")
    print(f"  McNemar test: p = {result.pvalue:.4f}"
          + (" *" if result.pvalue < 0.05 else ""))
    print()

In [4]:
# RUN STATISTICS

# ─────────────────────────────────────────────────────────────────
# RUN CLASSIFIERS — Alice → Fuglsang
# ─────────────────────────────────────────────────────────────────

print("Running Alice → Fuglsang classifiers...")

dec_alice_env   = {}
dec_alice_onset = {}
r_att_alice_env   = {}
r_ign_alice_env   = {}
r_att_alice_onset = {}
r_ign_alice_onset = {}

trf_env = eelbrain.load.unpickle(
    ALICE_TRF_DIR / f'64hz-universal-{helper_functions.get_trf_model_name(DATASET_TYPE.ALICE, PREDICTOR_TYPE.ENVELOPE, ATTENTION_TYPE.SINGLE, MODEL_TYPE.BACKWARD)}.pickle'
)
trf_onset = eelbrain.load.unpickle(
    ALICE_TRF_DIR / f'64hz-universal-{helper_functions.get_trf_model_name(DATASET_TYPE.ALICE, PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.SINGLE, MODEL_TYPE.BACKWARD)}.pickle'
)

for subject in FUGLSANG_SUBJECTS:
    eeg = eelbrain.load.unpickle(
        FUGLSANG_EEG_DIR / subject / f'{subject} eeg_trimmed.pickle'
    )
    true_att_env = eelbrain.load.unpickle(
        FUGLSANG_PREDICTOR_DIR / subject / f'attended_envelope_concat.pickle'
    ).x
    true_ign_env = eelbrain.load.unpickle(
        FUGLSANG_PREDICTOR_DIR / subject / f'ignored_envelope_concat.pickle'
    ).x
    true_att_onset = eelbrain.load.unpickle(
        FUGLSANG_PREDICTOR_DIR / subject / f'attended_envelope_onset_concat.pickle'
    ).x
    true_ign_onset = eelbrain.load.unpickle(
        FUGLSANG_PREDICTOR_DIR / subject / f'ignored_envelope_onset_concat.pickle'
    ).x

    dec_e, r_att_e, r_ign_e = helper_functions.aad_single_classifier(
        eeg, true_att_env,   true_ign_env,   trf_env)
    dec_o, r_att_o, r_ign_o = helper_functions.aad_single_classifier(
        eeg, true_att_onset, true_ign_onset, trf_onset)

    dec_alice_env[subject]     = dec_e
    dec_alice_onset[subject]   = dec_o
    r_att_alice_env[subject]   = r_att_e
    r_ign_alice_env[subject]   = r_ign_e
    r_att_alice_onset[subject] = r_att_o
    r_ign_alice_onset[subject] = r_ign_o

# ─────────────────────────────────────────────────────────────────
# RUN CLASSIFIERS — Fuglsang → SNHL
# (uncomment when SNHL data is ready)
# ─────────────────────────────────────────────────────────────────

# print("Running Fuglsang → SNHL classifiers...")
# acc_nh_env,   dec_nh_env,   r_att_nh_env,   r_ign_nh_env   = helper_functions.aad_classifier(
#     PREDICTOR_TYPE.ENVELOPE,       SNHL_SUBJECTS_NH,
#     generalised=GENERALISATION_TYPE.AVERAGE, cv=CROSS_VALIDATION_TYPE.HOLD_OUT,
#     aad_type=AAD_APPROACH.DOUBLE
# )
# acc_hi_env,   dec_hi_env,   r_att_hi_env,   r_ign_hi_env   = helper_functions.aad_classifier(
#     PREDICTOR_TYPE.ENVELOPE,       SNHL_SUBJECTS_HI,
#     generalised=GENERALISATION_TYPE.AVERAGE, cv=CROSS_VALIDATION_TYPE.HOLD_OUT,
#     aad_type=AAD_APPROACH.DOUBLE
# )
# acc_nh_onset, dec_nh_onset, r_att_nh_onset, r_ign_nh_onset = helper_functions.aad_classifier(
#     PREDICTOR_TYPE.ENVELOPE_ONSET, SNHL_SUBJECTS_NH,
#     generalised=GENERALISATION_TYPE.AVERAGE, cv=CROSS_VALIDATION_TYPE.HOLD_OUT,
#     aad_type=AAD_APPROACH.DOUBLE
# )
# acc_hi_onset, dec_hi_onset, r_att_hi_onset, r_ign_hi_onset = helper_functions.aad_classifier(
#     PREDICTOR_TYPE.ENVELOPE_ONSET, SNHL_SUBJECTS_HI,
#     generalised=GENERALISATION_TYPE.AVERAGE, cv=CROSS_VALIDATION_TYPE.HOLD_OUT,
#     aad_type=AAD_APPROACH.DOUBLE
# )

# ─────────────────────────────────────────────────────────────────
# 1. BINOMIAL TESTS
# ─────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("BINOMIAL TESTS — Classification accuracy vs chance (50%)")
print("=" * 60 + "\n")

report_binomial("Alice→Fuglsang — Envelope",
                sum(dec_alice_env.values()), len(FUGLSANG_SUBJECTS))
report_binomial("Alice→Fuglsang — Onset",
                sum(dec_alice_onset.values()), len(FUGLSANG_SUBJECTS))

# Uncomment when SNHL ready:
# report_binomial("Fuglsang→SNHL NH — Envelope",
#                 sum(dec_nh_env.values()), len(SNHL_SUBJECTS_NH))
# report_binomial("Fuglsang→SNHL HI — Envelope",
#                 sum(dec_hi_env.values()), len(SNHL_SUBJECTS_HI))
# report_binomial("Fuglsang→SNHL NH — Onset",
#                 sum(dec_nh_onset.values()), len(SNHL_SUBJECTS_NH))
# report_binomial("Fuglsang→SNHL HI — Onset",
#                 sum(dec_hi_onset.values()), len(SNHL_SUBJECTS_HI))

# ─────────────────────────────────────────────────────────────────
# 2. PAIRED T-TESTS
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("PAIRED T-TESTS — r_attended vs r_ignored")
print("=" * 60 + "\n")

report_ttest("Alice→Fuglsang — Envelope",
             r_att_alice_env,   r_ign_alice_env)
report_ttest("Alice→Fuglsang — Onset",
             r_att_alice_onset, r_ign_alice_onset)

# Uncomment when SNHL ready:
# report_ttest("Fuglsang→SNHL NH — Envelope", r_att_nh_env,   r_ign_nh_env)
# report_ttest("Fuglsang→SNHL HI — Envelope", r_att_hi_env,   r_ign_hi_env)
# report_ttest("Fuglsang→SNHL NH — Onset",    r_att_nh_onset, r_ign_nh_onset)
# report_ttest("Fuglsang→SNHL HI — Onset",    r_att_hi_onset, r_ign_hi_onset)

# ─────────────────────────────────────────────────────────────────
# 3. McNEMAR TESTS
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("McNEMAR TESTS — Envelope vs Onset")
print("=" * 60 + "\n")

report_mcnemar("Alice→Fuglsang: Envelope vs Onset",
               dec_alice_env, dec_alice_onset)

# Uncomment when SNHL ready:
# report_mcnemar("SNHL NH: Envelope vs Onset", dec_nh_env, dec_nh_onset)
# report_mcnemar("SNHL HI: Envelope vs Onset", dec_hi_env, dec_hi_onset)

# ─────────────────────────────────────────────────────────────────
# 4. BETWEEN-GROUP — NH vs HI (uncomment when SNHL ready)
# ─────────────────────────────────────────────────────────────────

# print("=" * 60)
# print("BETWEEN-GROUP TESTS — Normal-hearing vs Hearing-impaired")
# print("=" * 60 + "\n")

# report_ttest_between("r_attended NH vs HI — Envelope",
#                      r_att_nh_env, r_att_hi_env)
# report_ttest_between("r_attended NH vs HI — Onset",
#                      r_att_nh_onset, r_att_hi_onset)
# report_ttest_between("r_ignored NH vs HI — Envelope",
#                      r_ign_nh_env, r_ign_hi_env)
# report_ttest_between("r_ignored NH vs HI — Onset",
#                      r_ign_nh_onset, r_ign_hi_onset)

Running Alice → Fuglsang classifiers...

BINOMIAL TESTS — Classification accuracy vs chance (50%)

Alice→Fuglsang — Envelope
  Accuracy: 17/18 (94.4%)
  Binomial test vs chance: p = 0.0001 *

Alice→Fuglsang — Onset
  Accuracy: 16/18 (88.9%)
  Binomial test vs chance: p = 0.0007 *

PAIRED T-TESTS — r_attended vs r_ignored

Alice→Fuglsang — Envelope
  r_att: 0.0557 ± 0.0355
  r_ign: 0.0317 ± 0.0169
  Paired t-test: t = 3.376, p = 0.0036 *
  Cohen's d = 0.819

Alice→Fuglsang — Onset
  r_att: 0.0199 ± 0.0141
  r_ign: 0.0104 ± 0.0072
  Paired t-test: t = 3.579, p = 0.0023 *
  Cohen's d = 0.868

McNEMAR TESTS — Envelope vs Onset

Alice→Fuglsang: Envelope vs Onset
  Contingency table:
    [16, 1]
    [0, 1]
  McNemar test: p = 1.0000

